In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf

sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_elastic import NMF_logistic

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  # Restrict TensorFlow to only allocate 1GB of memory on the first GPU
  try:
    tf.config.experimental.set_virtual_device_configuration(
        gpus[0],
        [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=8192)])
    logical_gpus = tf.config.experimental.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Virtual devices must be set before GPUs have been initialized
    print(e)

sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)

indx_pos = (behaviornon1==1)&(condition==4)
indx_neg1 = (behaviornon1==2)&(condition==4)
indx_neg2 = (behaviornon1==2)&(condition==6)
indx_neg3 = (behaviornon1==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos

y = np.zeros(N)
y[indx_pos] = 1


In [ ]:
mouse = mouse[indx_tot]
group = group[indx_tot]
expDate = expDate[indx_tot]
behavior = behavior[indx_tot]
behaviornon1 = behaviornon1[indx_tot]
time = time[indx_tot]
condition = condition[indx_tot]
y = y[indx_tot]

N = len(mouse)

training_set_idx = np.ones(N)
training_set_idx[mouse=='Mouse048'] = 0
training_set_idx[mouse=='Mouse7980'] = 0
training_set_idx[mouse=='Mouse7998'] = 0

granger = np.exp(granger)
granger[granger>10] = 10
power = power*10
power[power>6] = 6

X = np.hstack((power,coherence,granger))
X = X[indx_tot]

print(mouse.shape)
print(X.shape)

X_train = X[training_set_idx==1]
m_train = mouse[training_set_idx==1]
y_train = y[training_set_idx==1]

X_test = X[training_set_idx==0]
m_test = mouse[training_set_idx==0]
y_test = y[training_set_idx==0]

In [ ]:
mu = 1.0

#>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# Add the newer data
power,coherence,granger,labels_new = load_data('/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_all_validate3.mat',fBounds=(1,56),feature_list=['power','coherence','granger'])

power = 10*power
power = power.astype(np.float32)
power[power>6] = 6


In [ ]:
coherence = coherence.astype(np.float32)
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

X_new = np.hstack((power,coherence,granger))

windows_new = labels_new['windows']
mouse_new = np.squeeze(windows_new['mouse'])
expDate_new = np.squeeze(windows_new['expDate'])
group_new = np.squeeze(windows_new['group'])
condition_new = np.squeeze(windows_new['condition'])
behavior_new = np.squeeze(windows_new['behavior'])
time_new = np.squeeze(windows_new['time'])

idx_pos_new = (condition_new==4)&(behavior_new==1)
indx_neg_new = (behavior_new==2)&((condition_new==4)|(condition_new==6)|(condition_new==8))
y_new = np.zeros(len(mouse_new))
y_new[idx_pos_new] = 1
idx_tot_new = idx_pos_new|indx_neg_new

X_new = X_new[idx_tot_new]
mouse_new = mouse_new[idx_tot_new]
y_new = y_new[idx_tot_new]


mice_new = np.unique(mouse_new)
nMice = len(mice_new)
mice_new_train = mice_new[:4]


ids = np.zeros(len(mouse_new))
for i in range(4):
    ids[mouse_new==mice_new_train[i]] = 1

print('>>>>>>>>>>>>>.')
print(y_new.shape)
print(ids.shape)
print(mouse_new.shape)
print(X_new.shape)

X_train_new = X_new[ids==1,:]
X_test_new = X_new[ids==0,:]
y_train_new = y_new[ids==1]
y_test_new = y_new[ids==0]

print(X_train_new.shape)
print(y_train_new.shape)
print(X_test_new.shape)
print(y_test_new.shape)


In [ ]:
X_train_new = X_new[ids==1,:]
X_test_new = X_new[ids==0,:]
y_train_new = y_new[ids==1]
y_test_new = y_new[ids==0]

print(X_train_new.shape)
print(y_train_new.shape)
print(X_test_new.shape)
print(y_test_new.shape)

#>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

X_train_tot = np.vstack((X_train,X_train_new))
y_train_tot = np.concatenate((y_train,y_train_new))
weights_g = np.ones(X_train_tot.shape[0])
weights_s = np.ones(X_train_tot.shape[0])

nFact = 8


In [ ]:
myDict = pickle.load(open('../Unbalanced_Elastic_12_enc_1.0.p','rb'))
myDict.keys()

In [ ]:
A_enc = myDict['A_enc']
B_enc = myDict['B_enc']

sys.path.append('/home/austin/Basic')
from utils_np import safe_softplus

In [ ]:
S_train = myDict['S_train']
S_test = myDict['S_test']
S_train_new = myDict['S_train_new']
S_test_new = myDict['S_test_new']

In [ ]:
np.mean(S_train[:,0])

In [ ]:
sns.distplot(S_train[:,0],hist=False)

In [ ]:
sns.distplot(S_train_new[:,0],hist=False)

In [ ]:
sns.distplot(S_test[:,0],hist=False)

In [ ]:
sns.distplot(S_test_new[:,0],hist=False)

In [ ]:
quantile_train_25 = np.quantile(S_train[:,0],.25)
print(quantile_train)

In [ ]:
quantile_train_1 = np.quantile(S_train[:,0],.1)
print(quantile_train)

In [ ]:
idx_choice = quantile_train_25
idx_train = (S_train[:,0]<idx_choice)
idx_test = (S_test[:,0]<idx_choice)
idx_train_new = (S_train_new[:,0]<idx_choice)
idx_test_new = (S_test_new[:,0]<idx_choice)

In [ ]:
X_sub_tr = X_train[idx_train]
X_sub_te = X_test[idx_test]
Xn_sub_tr = X_train_new[idx_train_new]
Xn_sub_te = X_test_new[idx_test_new]

Y_sub_tr = y_train[idx_train]
Y_sub_te = y_test[idx_test]
Yn_sub_tr = y_train_new[idx_train_new]
Yn_sub_te = y_test_new[idx_test_new]


In [ ]:
mm_new = np.zeros(len(mouse_new))
for i in range(len(mouse_new)):
    mm_new[i] = np.where(mouse_new[i]==mice_new)[0][0]

mm_train = mm_new[ids==1]
mm_test = mm_new[ids==0]

In [ ]:
mm_sub_te = mm_test[idx_test_new]

## Now actually fit model on training data